In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import SMOTE

In [2]:
# Load the data
train_data_path = 'train_data.csv'
test_data_path = 'test_data.csv'

train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)

# Check for missing values
missing_values = train_data.isnull().sum()
print("\nMissing Values in Each Column:\n", missing_values[missing_values > 0])


Missing Values in Each Column:
 Revenue                        384
Revenue Growth                 520
Cost of Revenue                495
Gross Profit                   391
R&D Expenses                   562
                              ... 
Asset Growth                   563
Book Value per Share Growth    636
Debt Growth                    608
R&D Expense Growth             605
SG&A Expenses Growth           591
Length: 221, dtype: int64


In [3]:
# Identify columns to drop (those with > 50% missing values)
columns_to_drop = ['Name', 'Sector', 'cashConversionCycle', 'operatingCycle', 'shortTermCoverageRatios']
columns_to_drop = [col for col in columns_to_drop if col in train_data.columns]

# Drop columns from both train and test data
X_train = train_data.drop(columns=columns_to_drop + ['Class'])
y_train = train_data['Class']
X_test = test_data.drop(columns=columns_to_drop)

# Impute missing values with the median
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
# Apply SMOTE to balance the classes
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

In [5]:
# Define the best parameters from hyperparameter tuning
xgb_best_params = {
    'subsample': 0.8,
    'n_estimators': 100,
    'max_depth': 10,
    'learning_rate': 0.01,
    'gamma': 0,
    'colsample_bytree': 1.0
}

# Train the XGBoost model using the best parameters
xgb_model = XGBClassifier(**xgb_best_params, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_balanced, y_train_balanced)

# Make predictions on the validation set
X_val, y_val = X_train_scaled, y_train  # Use original validation data for evaluation
xgb_preds = xgb_model.predict(X_val)

# Evaluate the model
print("\nXGBoost with Tuned Parameters:")
print("Accuracy:", accuracy_score(y_val, xgb_preds))
print(classification_report(y_val, xgb_preds))


XGBoost with Tuned Parameters:
Accuracy: 0.9502746458514021
              precision    recall  f1-score   support

           0       0.97      0.96      0.97      2498
           1       0.91      0.92      0.91       961

    accuracy                           0.95      3459
   macro avg       0.94      0.94      0.94      3459
weighted avg       0.95      0.95      0.95      3459



In [7]:
# Make predictions on the test data
test_preds = xgb_model.predict(X_test_scaled)

# Prepare the submission DataFrame
submission = pd.DataFrame({
    'Name': test_data['Name'],
    'Class': test_preds
})

# Save the predictions to a CSV file
submission.to_csv('submission.csv', index=False)
print("\nSubmission file saved as submission.csv")


Submission file saved as submission.csv


In [10]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(xgb_model, X_train_balanced, y_train_balanced, cv=skf, scoring='accuracy')
print("Cross-Validation Accuracy Scores:", scores)
print("Mean Accuracy:", np.mean(scores))

Cross-Validation Accuracy Scores: [0.755      0.78078078 0.77877878 0.78278278 0.75775776]
Mean Accuracy: 0.77102002002002
